# Tugas Praktikum 4: NoSQL dan Pemodelan Data

**Mata Kuliah:** Praktikum Big Data  
**Nama:** Arya Pratama Hendri  
**NIM:** 2411537001  
**Program Studi:** Informatika, Fakultas Teknologi Informasi, Universitas Andalas

Notebook ini mengerjakan Tugas Praktikum pada Bagian R modul. R-1 merekap benchmark, R-2 menyimpan data lain dengan tiga cara, dan R-3 mensimulasikan tabel Delta selama dua hari kerja. Rancangan penyimpanan studi kasus (R-4) ada di laporan PDF.

Modul tidak memberi kode untuk tugas, jadi kode di notebook ini saya tulis sendiri dengan memakai fungsi dan pustaka yang sama seperti di praktikum. Datanya tetap data tiruan dari pembangkit K-1b, karena folder `trips_bersih/` Praktikum 3 tidak ada di Drive saya.

## Persiapan

Notebook ini berjalan di runtime baru, jadi pustaka perlu dipasang lagi dan data tiruan perlu dibuat ulang. Fungsi pembangkitnya sama dengan K-1b di modul dan memakai seed 42, sehingga isinya sama dengan data di notebook praktikum. File `benchmark_nosql.csv` dibaca dari folder Praktikum4 hasil K-9, jadi notebook praktikum harus sudah dijalankan lebih dulu.

In [1]:
!pip install -q duckdb networkx shapely tinydb "deltalake>=1.0"

from google.colab import drive
drive.mount("/content/drive")

import os, time, json, sqlite3, shutil
import numpy as np, pandas as pd

DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4"
print("benchmark_nosql.csv tersedia?", os.path.exists(f"{DIR_SIMPAN}/benchmark_nosql.csv"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 31.4 MB/s eta 0:00:00
Mounted at /content/drive
benchmark_nosql.csv tersedia? True


In [2]:
def bangkitkan_trips_sintetis(keluar, n=1_000_000, seed=42):
    rng = np.random.default_rng(seed)
    wilayah = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
    zona_ids = np.arange(1, 264)
    wilayah_zona = rng.choice(wilayah, size=len(zona_ids), p=[0.35, 0.25, 0.22, 0.10, 0.05, 0.03])
    zona = pd.DataFrame({"LocationID": zona_ids, "borough_naik": wilayah_zona})

    pop = np.clip(rng.zipf(1.6, size=len(zona_ids)).astype(float), 1, 60)
    pop /= pop.sum()
    pu = rng.choice(zona_ids, size=n, p=pop)
    do = rng.choice(zona_ids, size=n, p=pop)

    awal = pd.Timestamp("2023-01-01")
    pickup = awal + pd.to_timedelta(rng.integers(0, 31 * 24 * 60, n), unit="m")
    durasi = rng.integers(2, 90, n)
    dist = np.round(rng.gamma(2.0, 1.6, n).clip(0.05, 60), 2)
    fare = np.round(2.5 + dist * 2.8 + rng.gamma(2.0, 1.0, n), 2)
    tip = np.round(rng.gamma(1.5, 1.2, n), 2)
    pay = rng.choice([1, 2, 3, 4], size=n, p=[0.7, 0.27, 0.02, 0.01])

    d = pd.DataFrame({
        "tpep_pickup_datetime": pickup,
        "tpep_dropoff_datetime": pickup + pd.to_timedelta(durasi, unit="m"),
        "passenger_count": rng.integers(1, 5, n).astype("float64"),
        "trip_distance": dist, "PULocationID": pu, "DOLocationID": do,
        "payment_type": pay, "fare_amount": fare, "tip_amount": tip,
        "total_amount": np.round(fare + tip + 3.0, 2),
        "durasi_menit": durasi.astype("float64"),
        "nama_pembayaran": pd.Series(pay).map({1: "Kartu kredit", 2: "Tunai", 3: "Gratis", 4: "Sengketa"}).values,
    })
    d = d.merge(zona.rename(columns={"LocationID": "PULocationID"}), on="PULocationID", how="left")
    d["suhu_c"] = np.round(rng.normal(3, 5, n), 1)
    d["hujan_mm"] = np.round(rng.gamma(0.4, 0.8, n), 2)
    if os.path.exists(keluar):
        shutil.rmtree(keluar)
    d.to_parquet(keluar, partition_cols=["borough_naik"], index=False)
    return keluar

TRIPS_BERSIH = bangkitkan_trips_sintetis("/content/trips_bersih_sintetis")

NAMA_BARU = {
    "tpep_pickup_datetime": "waktu_jemput",
    "PULocationID": "zona_jemput",
    "DOLocationID": "zona_tujuan",
    "borough_naik": "wilayah_jemput",
    "trip_distance": "jarak_mil",
    "total_amount": "total_bayar",
    "tip_amount": "tip",
    "nama_pembayaran": "metode_bayar",
}
df = pd.read_parquet(TRIPS_BERSIH).rename(columns=NAMA_BARU)
df["wilayah_jemput"] = df["wilayah_jemput"].astype(str)
print("baris:", f"{len(df):,}", "| kolom:", df.shape[1])

baris: 1,000,000 | kolom: 15


## R-1. Rekap Benchmark

Sel pertama membaca `benchmark_nosql.csv` dan menyusun tabel rekap K-9 dari angka di dalamnya. Sel kedua menulis satu kalimat tafsiran untuk setiap baris benchmark. Kalimat-kalimatnya dibentuk dari angka di file, jadi isinya ikut menyesuaikan bila notebook praktikum dijalankan ulang.

In [3]:
bm = pd.read_csv(f"{DIR_SIMPAN}/benchmark_nosql.csv")
t = dict(zip(bm["langkah"], bm["detik"]))
kali = lambda a, b: f"{t[a] / t[b]:,.1f} kali"

loop_1000 = t["cara 1 - loop Python, 5 kotak"] / 5 * 1000          # perkiraan, loop hanya dijalankan 5 kotak
str_total = t["bangun indeks STRtree"] + t["cara 3 - STRtree, 1000 kotak"]

rekap_k9 = pd.DataFrame([
    ["Ambil satu data lewat id", "Key-value; tabel dengan indeks", "Tabel tanpa indeks",
     f"tanpa indeks {t['cari 200 perjalanan - SQL tanpa indeks']} s, berindeks {t['cari 200 perjalanan - SQL dengan indeks']} s, key-value {t['cari 200 perjalanan - key-value']} s"],
    ["Menghitung atau merangkum banyak baris", "Per kolom (DuckDB pada Parquet)", "Key-value (buka semua nilai)",
     f"DuckDB {t['DuckDB - hitung per wilayah (langsung dari Parquet)']} s, SQLite {t['SQLite - hitung per wilayah']} s, key-value {t['hitung per wilayah - key-value (buka semua nilai)']} s"],
    ["Data dengan field opsional atau berubah-ubah", "Document", "Tabel (kolom NULL, migrasi)", "kualitatif"],
    ["Keterhubungan dengan jumlah langkah bebas", "Graph", "SQL (satu JOIN per langkah)",
     f"graph {t['2 langkah - graph (NetworkX)']} s, SQL {t['2 langkah - SQL self-join (DuckDB)']} s"],
    ["Mencari lokasi dalam wilayah", "Spasial dengan indeks", "Memeriksa semua titik",
     f"loop sekitar {loop_1000:.1f} s (perkiraan), NumPy {t['cara 2 - NumPy, 1000 kotak']} s, STRtree {str_total:.4f} s"],
    ["Perubahan yang harus aman dan bisa dibatalkan", "Tabel Delta", "Folder Parquet biasa", "kualitatif"],
], columns=["Kebutuhan", "Cara yang unggul", "Cara yang lemah", "Angka saya"])
pd.set_option("display.max_colwidth", None)
rekap_k9

,Kebutuhan,Cara yang unggul,Cara yang lemah,Angka saya
0,Ambil satu data lewat id,Key-value; tabel dengan indeks,Tabel tanpa indeks,"tanpa indeks 4.6484 s, berindeks 0.0043 s, key-value 0.0013 s"
1,Menghitung atau merangkum banyak baris,Per kolom (DuckDB pada Parquet),Key-value (buka semua nilai),"DuckDB 0.0364 s, SQLite 0.8231 s, key-value 1.2916 s"
2,Data dengan field opsional atau berubah-ubah,Document,"Tabel (kolom NULL, migrasi)",kualitatif
3,Keterhubungan dengan jumlah langkah bebas,Graph,SQL (satu JOIN per langkah),"graph 0.0135 s, SQL 0.0184 s"
4,Mencari lokasi dalam wilayah,Spasial dengan indeks,Memeriksa semua titik,"loop sekitar 2584.2 s (perkiraan), NumPy 0.5473 s, STRtree 0.2913 s"
5,Perubahan yang harus aman dan bisa dibatalkan,Tabel Delta,Folder Parquet biasa,kualitatif


In [4]:
tafsir = {
    "baca trips_bersih": "Satu juta baris Parquet terbaca ke pandas dalam waktu singkat.",
    "relasional: muat tabel": "SQLite butuh waktu ini untuk menyalin 300.000 baris ke tabel.",
    "key-value: bangun dict": f"Menulis 300.000 teks JSON {kali('key-value: bangun dict', 'relasional: muat tabel')} lebih lama daripada memuat tabel SQLite.",
    "cari 200 perjalanan - SQL tanpa indeks": "Tanpa indeks SQLite memeriksa baris satu per satu untuk setiap id.",
    "cari 200 perjalanan - SQL dengan indeks": f"Indeks mempercepat pencarian {kali('cari 200 perjalanan - SQL tanpa indeks', 'cari 200 perjalanan - SQL dengan indeks')}.",
    "cari 200 perjalanan - key-value": f"Kunci langsung menunjuk data, {kali('cari 200 perjalanan - SQL dengan indeks', 'cari 200 perjalanan - key-value')} lebih cepat dari tabel berindeks.",
    "hitung per wilayah - SQL GROUP BY": "SQLite mengenal kolom total_bayar sehingga bisa langsung menghitungnya.",
    "hitung per wilayah - key-value (buka semua nilai)": f"Membuka semua nilai JSON membuat key-value {kali('hitung per wilayah - key-value (buka semua nilai)', 'hitung per wilayah - SQL GROUP BY')} lebih lambat dari GROUP BY.",
    "DuckDB - hitung per wilayah (langsung dari Parquet)": f"DuckDB hanya membaca dua kolom dan {kali('SQLite - hitung per wilayah', 'DuckDB - hitung per wilayah (langsung dari Parquet)')} lebih cepat dari SQLite.",
    "SQLite - muat seluruh baris ke tabel": "SQLite perlu langkah muat ini, sedangkan DuckDB membaca Parquet tanpa memuat.",
    "SQLite - hitung per wilayah": "Penyimpanan per baris ikut membaca kolom yang tidak dipakai query.",
    "DuckDB - hanya wilayah Staten Island": "Filter pada kolom partisi membuat DuckDB hanya membuka satu folder.",
    "hitung arus antar zona": "DuckDB meringkas satu juta perjalanan menjadi tabel arus antarzona.",
    "2 langkah - graph (NetworkX)": "Graph menelusuri tetangga dari tetangga untuk menjawab dua langkah.",
    "2 langkah - SQL self-join (DuckDB)": "Self-join memberi hasil yang sama dengan waktu yang berdekatan dengan graph.",
    "cara 1 - loop Python, 5 kotak": f"Lima kotak saja sudah memakan waktu ini, jadi 1.000 kotak kira-kira {loop_1000:.0f} detik.",
    "cara 2 - NumPy, 1000 kotak": "NumPy memeriksa semua titik sekaligus tanpa indeks dan menjadi pembanding yang adil.",
    "bangun indeks STRtree": "Biaya membangun indeks dibayar satu kali di awal.",
    "cara 3 - STRtree, 1000 kotak": f"STRtree melewati kelompok titik yang jauh dari kotak, {t['cara 2 - NumPy, 1000 kotak'] / str_total:.1f} kali lebih cepat dari NumPy termasuk waktu bangun.",
}
bm["tafsiran"] = bm["langkah"].map(tafsir)
bm

,langkah,detik,tafsiran
0,baca trips_bersih,0.9373,Satu juta baris Parquet terbaca ke pandas dalam waktu singkat.
1,relasional: muat tabel,2.8616,SQLite butuh waktu ini untuk menyalin 300.000 baris ke tabel.
2,key-value: bangun dict,4.9502,Menulis 300.000 teks JSON 1.7 kali lebih lama daripada memuat tabel SQLite.
3,cari 200 perjalanan - SQL tanpa indeks,4.6484,Tanpa indeks SQLite memeriksa baris satu per satu untuk setiap id.
4,cari 200 perjalanan - SQL dengan indeks,0.0043,"Indeks mempercepat pencarian 1,081.0 kali."
5,cari 200 perjalanan - key-value,0.0013,"Kunci langsung menunjuk data, 3.3 kali lebih cepat dari tabel berindeks."
6,hitung per wilayah - SQL GROUP BY,0.2279,SQLite mengenal kolom total_bayar sehingga bisa langsung menghitungnya.
7,hitung per wilayah - key-value (buka semua nilai),1.2916,Membuka semua nilai JSON membuat key-value 5.7 kali lebih lambat dari GROUP BY.
8,DuckDB - hitung per wilayah (langsung dari Parquet),0.0364,DuckDB hanya membaca dua kolom dan 22.6 kali lebih cepat dari SQLite.
9,SQLite - muat seluruh baris ke tabel,2.5006,"SQLite perlu langkah muat ini, sedangkan DuckDB membaca Parquet tanpa memuat."


## R-2. Data Lain, Cara Lain

Data yang saya pilih adalah **hari**. Setiap tanggal di bulan Januari 2023 memiliki jumlah perjalanan, total pendapatan, rata-rata jarak, serta rincian per metode bayar dan per wilayah. Ringkasan harian ini saya simpan dengan tiga cara: tabel (SQLite), document (TinyDB), dan key-value (kamus Python). Untuk setiap cara ada satu pertanyaan yang mudah dan satu pertanyaan yang canggung, dan waktunya saya ukur.

In [5]:
df["tanggal"] = df["waktu_jemput"].dt.strftime("%Y-%m-%d")

harian = (df.groupby("tanggal")
            .agg(jumlah=("total_bayar", "size"),
                 pendapatan=("total_bayar", "sum"),
                 rata_jarak=("jarak_mil", "mean"))
            .round(2).reset_index())
harian["nama_hari"] = pd.to_datetime(harian["tanggal"]).dt.day_name()

per_metode = df.groupby(["tanggal", "metode_bayar"]).size().reset_index(name="jumlah")
per_wilayah = df.groupby(["tanggal", "wilayah_jemput"])["total_bayar"].sum().round(2).reset_index(name="pendapatan")

print("jumlah hari:", len(harian))
harian.head()

jumlah hari: 31


,tanggal,jumlah,pendapatan,rata_jarak,nama_hari
0,2023-01-01,32309,590498.01,3.21,Sunday
1,2023-01-02,32204,587472.55,3.19,Monday
2,2023-01-03,32226,589394.96,3.21,Tuesday
3,2023-01-04,32613,596930.32,3.21,Wednesday
4,2023-01-05,32009,586927.79,3.23,Thursday


### Cara 1: tabel (SQLite)

Ringkasan harian, rincian metode bayar, dan rincian wilayah masing-masing menjadi satu tabel. Pertanyaan mudah: rata-rata pendapatan untuk setiap nama hari, cukup satu `GROUP BY`. Pertanyaan canggung: profil lengkap satu tanggal, karena datanya tersebar di tiga tabel dan harus diambil dengan tiga query lalu dirangkai sendiri.

In [6]:
if os.path.exists("hari.db"):
    os.remove("hari.db")
con_h = sqlite3.connect("hari.db")
harian.to_sql("hari", con_h, index=False)
per_metode.to_sql("hari_metode", con_h, index=False)
per_wilayah.to_sql("hari_wilayah", con_h, index=False)

# Mudah: satu GROUP BY
t0 = time.perf_counter()
mudah_tabel = con_h.execute("""SELECT nama_hari, ROUND(AVG(pendapatan), 2) AS rata_pendapatan
                               FROM hari GROUP BY nama_hari ORDER BY rata_pendapatan DESC""").fetchall()
print(f"[mudah] rata-rata pendapatan per nama hari: {time.perf_counter() - t0:.4f} s")
for baris in mudah_tabel:
    print("  ", baris)

# Canggung: satu tanggal tersebar di tiga tabel
TANGGAL = "2023-01-15"
t0 = time.perf_counter()
profil = dict(zip(["tanggal", "jumlah", "pendapatan", "rata_jarak", "nama_hari"],
                  con_h.execute("SELECT * FROM hari WHERE tanggal = ?", (TANGGAL,)).fetchone()))
profil["per_metode"] = dict(con_h.execute("SELECT metode_bayar, jumlah FROM hari_metode WHERE tanggal = ?", (TANGGAL,)).fetchall())
profil["per_wilayah"] = dict(con_h.execute("SELECT wilayah_jemput, pendapatan FROM hari_wilayah WHERE tanggal = ?", (TANGGAL,)).fetchall())
print(f"[canggung] profil lengkap {TANGGAL} dari tiga tabel: {time.perf_counter() - t0:.4f} s")
print(profil)

[mudah] rata-rata pendapatan per nama hari: 0.0005 s
   ('Wednesday', 591413.15)
   ('Friday', 591137.53)
   ('Monday', 589859.74)
   ('Tuesday', 588880.66)
   ('Saturday', 588555.58)
   ('Thursday', 587846.02)
   ('Sunday', 587577.24)
[canggung] profil lengkap 2023-01-15 dari tiga tabel: 0.0008 s
{'tanggal': '2023-01-15', 'jumlah': 32191, 'pendapatan': 589250.43, 'rata_jarak': 3.22, 'nama_hari': 'Sunday', 'per_metode': {'Gratis': 621, 'Kartu kredit': 22498, 'Sengketa': 330, 'Tunai': 8742}, 'per_wilayah': {'Bronx': 34484.54, 'Brooklyn': 121066.89, 'EWR': 2202.79, 'Manhattan': 233125.73, 'Queens': 158574.26, 'Staten Island': 39796.22}}


### Cara 2: document (TinyDB)

Setiap tanggal menjadi satu dokumen, dan rincian metode bayar serta wilayah ditanam di dalamnya (embedding). Pertanyaan mudah: profil lengkap satu tanggal, cukup satu kali baca. Pertanyaan canggung: total pendapatan sebulan untuk setiap wilayah, karena angkanya tersebar di dalam 31 dokumen sehingga semua dokumen harus dibuka dan dijumlahkan sendiri.

In [7]:
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

def ke_dokumen_hari(r):
    return {
        "tanggal": r.tanggal,
        "nama_hari": r.nama_hari,
        "ringkasan": {"jumlah": int(r.jumlah), "pendapatan": float(r.pendapatan), "rata_jarak": float(r.rata_jarak)},
        "per_metode": {m: int(j) for m, j in per_metode.loc[per_metode["tanggal"] == r.tanggal, ["metode_bayar", "jumlah"]].values},
        "per_wilayah": {w: float(p) for w, p in per_wilayah.loc[per_wilayah["tanggal"] == r.tanggal, ["wilayah_jemput", "pendapatan"]].values},
    }

db_h = TinyDB(storage=MemoryStorage)
tabel_hari = db_h.table("hari")
tabel_hari.insert_multiple([ke_dokumen_hari(r) for r in harian.itertuples()])
print("jumlah dokumen:", len(tabel_hari))

# Mudah: satu dokumen sudah berisi semuanya
H = Query()
t0 = time.perf_counter()
dok = tabel_hari.get(H.tanggal == TANGGAL)
print(f"[mudah] profil lengkap {TANGGAL}: {time.perf_counter() - t0:.4f} s")
print(json.dumps(dok, indent=2))

# Canggung: total per wilayah harus dirakit dari isi semua dokumen
t0 = time.perf_counter()
total_wilayah = {}
for d in tabel_hari.all():
    for w, p in d["per_wilayah"].items():
        total_wilayah[w] = round(total_wilayah.get(w, 0) + p, 2)
print(f"[canggung] total pendapatan sebulan per wilayah: {time.perf_counter() - t0:.4f} s")
print(sorted(total_wilayah.items(), key=lambda kv: -kv[1]))

jumlah dokumen: 31
[mudah] profil lengkap 2023-01-15: 0.0003 s
{
  "tanggal": "2023-01-15",
  "nama_hari": "Sunday",
  "ringkasan": {
    "jumlah": 32191,
    "pendapatan": 589250.43,
    "rata_jarak": 3.22
  },
  "per_metode": {
    "Gratis": 621,
    "Kartu kredit": 22498,
    "Sengketa": 330,
    "Tunai": 8742
  },
  "per_wilayah": {
    "Bronx": 34484.54,
    "Brooklyn": 121066.89,
    "EWR": 2202.79,
    "Manhattan": 233125.73,
    "Queens": 158574.26,
    "Staten Island": 39796.22
  }
}
[canggung] total pendapatan sebulan per wilayah: 0.0008 s
[('Manhattan', 7141668.67), ('Queens', 4950607.71), ('Brooklyn', 3757865.14), ('Staten Island', 1225403.88), ('Bronx', 1109006.89), ('EWR', 82844.99)]


### Cara 3: key-value (kamus Python)

Kuncinya berbentuk `hari:<tanggal>` dan nilainya teks JSON dari dokumen yang sama. Pertanyaan mudah: mengambil satu tanggal lewat kuncinya. Pertanyaan canggung: mencari tanggal yang pendapatannya di atas rata-rata bulanan, karena penyimpanan key-value tidak bisa menyaring berdasarkan isi nilai sehingga semua nilai harus dibuka.

In [8]:
kv_hari = {f"hari:{d['tanggal']}": json.dumps(d) for d in tabel_hari.all()}
print("jumlah kunci:", len(kv_hari))

# Mudah: langsung lewat kunci
t0 = time.perf_counter()
satu_hari = json.loads(kv_hari[f"hari:{TANGGAL}"])
print(f"[mudah] ambil {TANGGAL} lewat kunci: {time.perf_counter() - t0:.6f} s | pendapatan:", satu_hari["ringkasan"]["pendapatan"])

# Canggung: menyaring berdasarkan isi, semua nilai dibuka satu per satu
t0 = time.perf_counter()
semua_nilai = [json.loads(v) for v in kv_hari.values()]
rata_bulan = sum(d["ringkasan"]["pendapatan"] for d in semua_nilai) / len(semua_nilai)
di_atas = sorted(d["tanggal"] for d in semua_nilai if d["ringkasan"]["pendapatan"] > rata_bulan)
print(f"[canggung] tanggal dengan pendapatan di atas rata-rata ({rata_bulan:,.2f}): {time.perf_counter() - t0:.4f} s")
print(len(di_atas), "tanggal:", di_atas)

# Silang dengan SQL supaya yakin jawabannya sama
dari_sql = [r[0] for r in con_h.execute("SELECT tanggal FROM hari WHERE pendapatan > (SELECT AVG(pendapatan) FROM hari) ORDER BY tanggal")]
print("sama dengan SQL?", di_atas == dari_sql)

jumlah kunci: 31
[mudah] ambil 2023-01-15 lewat kunci: 0.000315 s | pendapatan: 589250.43
[canggung] tanggal dengan pendapatan di atas rata-rata (589,270.88): 0.0019 s
14 tanggal: ['2023-01-01', '2023-01-03', '2023-01-04', '2023-01-09', '2023-01-10', '2023-01-13', '2023-01-16', '2023-01-17', '2023-01-19', '2023-01-20', '2023-01-23', '2023-01-25', '2023-01-27', '2023-01-28']
sama dengan SQL? True


Ringkasan R-2:

| Cara menyimpan data hari | Pertanyaan yang mudah | Pertanyaan yang canggung |
|---|---|---|
| Tabel (SQLite) | Rata-rata pendapatan per nama hari, satu GROUP BY | Profil lengkap satu tanggal, tersebar di tiga tabel dan butuh tiga query |
| Document (TinyDB) | Profil lengkap satu tanggal, satu kali baca | Total pendapatan sebulan per wilayah, semua dokumen harus dibuka |
| Key-value (kamus) | Ambil satu tanggal lewat kunci `hari:<tanggal>` | Tanggal dengan pendapatan di atas rata-rata, semua nilai harus dibuka |

## R-3. Delta Lake Dua Hari

Simulasi ini memakai perjalanan tanggal 1 dan 2 Januari 2023. Pada hari pertama, data tanggal 1 ditulis ke tabel Delta. Pada hari kedua, data tanggal 2 ditambahkan, lalu beberapa baris dikoreksi: perjalanan dengan metode bayar Gratis seharusnya tercatat dengan total bayar 0. Setelah itu tabel dibaca kembali seperti keadaannya di akhir hari pertama.

Setiap potongan data diberi `reset_index(drop=True)`. Tanpa itu, pandas ikut menulis indeks hasil filter sebagai kolom tambahan, sehingga data hari kedua akan ditolak karena jumlah kolomnya berbeda.

In [10]:
from deltalake import DeltaTable, write_deltalake

KOLOM_DELTA = ["waktu_jemput", "zona_jemput", "zona_tujuan",
               "wilayah_jemput", "jarak_mil", "total_bayar", "metode_bayar"]
hari_1 = df.loc[df["tanggal"] == "2023-01-01", KOLOM_DELTA].sort_values("waktu_jemput").reset_index(drop=True)
hari_2 = df.loc[df["tanggal"] == "2023-01-02", KOLOM_DELTA].sort_values("waktu_jemput").reset_index(drop=True)
print("baris tanggal 1:", len(hari_1), "| baris tanggal 2:", len(hari_2))

# Memindahkan path penyimpanan dari Google Drive ke lokal runtime Colab karena Google Drive FUSE tidak mendukung POSIX rename
PATH_DUA_HARI = "/content/trips_delta_dua_hari"
if os.path.exists(PATH_DUA_HARI):
    shutil.rmtree(PATH_DUA_HARI)

# HARI PERTAMA: tulis data tanggal 1
write_deltalake(PATH_DUA_HARI, hari_1, mode="overwrite")
VERSI_HARI_1 = DeltaTable(PATH_DUA_HARI).version()
print("akhir hari pertama -> versi:", VERSI_HARI_1, "| baris:", len(DeltaTable(PATH_DUA_HARI).to_pandas()))

baris tanggal 1: 32309 | baris tanggal 2: 32204
akhir hari pertama -> versi: 0 | baris: 32309


In [11]:
# HARI KEDUA (1): tambah data tanggal 2
write_deltalake(PATH_DUA_HARI, hari_2, mode="append")
dt2 = DeltaTable(PATH_DUA_HARI)
print("setelah tambah data -> versi:", dt2.version(), "| baris:", len(dt2.to_pandas()))

# HARI KEDUA (2): koreksi, perjalanan Gratis seharusnya bertotal 0
gabungan = pd.concat([hari_1, hari_2])
n_gratis = int((gabungan["metode_bayar"] == "Gratis").sum())
n_gratis_h1 = int((hari_1["metode_bayar"] == "Gratis").sum())
print("baris Gratis yang dikoreksi:", n_gratis, "| di antaranya dari tanggal 1:", n_gratis_h1)
dt2.update(predicate="metode_bayar = 'Gratis'", updates={"total_bayar": "0.0"})
dt2 = DeltaTable(PATH_DUA_HARI)
print("setelah koreksi -> versi:", dt2.version(), "| baris:", len(dt2.to_pandas()))

setelah tambah data -> versi: 1 | baris: 64513
baris Gratis yang dikoreksi: 1275 | di antaranya dari tanggal 1: 650
setelah koreksi -> versi: 2 | baris: 64513


In [12]:
# Membaca tabel seperti keadaannya di akhir hari pertama
akhir_hari_1 = DeltaTable(PATH_DUA_HARI, version=VERSI_HARI_1).to_pandas()
sekarang = DeltaTable(PATH_DUA_HARI).to_pandas()

print("baris akhir hari pertama:", len(akhir_hari_1), "| baris sekarang:", len(sekarang))
print("waktu jemput terakhir pada akhir hari pertama:", akhir_hari_1["waktu_jemput"].max())
urut = lambda d: d.sort_values(KOLOM_DELTA).reset_index(drop=True).astype(str)
print("isi akhir hari pertama sama dengan data tanggal 1?", urut(akhir_hari_1).equals(urut(hari_1)))

# Koreksi hari kedua mengubah baris tanggal 1 di versi terbaru, tetapi tidak di versi akhir hari pertama
gratis_lama = akhir_hari_1.loc[akhir_hari_1["metode_bayar"] == "Gratis", "total_bayar"].sum()
gratis_baru = sekarang.loc[(sekarang["metode_bayar"] == "Gratis") & (sekarang["waktu_jemput"] < "2023-01-02"), "total_bayar"].sum()
print(f"total_bayar perjalanan Gratis tanggal 1 -> akhir hari pertama: {gratis_lama:,.2f} | sekarang: {gratis_baru:,.2f}")

baris akhir hari pertama: 32309 | baris sekarang: 64513
waktu jemput terakhir pada akhir hari pertama: 2023-01-01 23:59:00
isi akhir hari pertama sama dengan data tanggal 1? True
total_bayar perjalanan Gratis tanggal 1 -> akhir hari pertama: 12,029.34 | sekarang: 0.00


In [13]:
# Riwayat tabel dan isi _delta_log/
for h in DeltaTable(PATH_DUA_HARI).history():
    print("versi", h.get("version"), "|", h.get("operation"), "|", h.get("operationParameters"))

LOG = f"{PATH_DUA_HARI}/_delta_log"
print("\nisi _delta_log/:", sorted(os.listdir(LOG)))
for berkas in sorted(f for f in os.listdir(LOG) if f.endswith(".json")):
    print(f"\n--- {berkas}")
    with open(f"{LOG}/{berkas}") as fh:
        for baris in fh:
            aksi, isi = next(iter(json.loads(baris).items()))
            if aksi in ("add", "remove"):
                print(f"  {aksi:<10} {isi['path']}")
            elif aksi == "commitInfo":
                print(f"  {aksi:<10} operation={isi.get('operation')}")
            else:
                print(f"  {aksi:<10} {json.dumps(isi)[:90]}")

versi 2 | UPDATE | {'predicate': "metode_bayar = 'Gratis'"}
versi 1 | WRITE | {'mode': 'Append'}
versi 0 | WRITE | {'mode': 'Overwrite'}

isi _delta_log/: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json']

--- 00000000000000000000.json
  commitInfo operation=WRITE
  protocol   {"minReaderVersion": 3, "minWriterVersion": 7, "readerFeatures": ["timestampNtz"], "writer
  metaData   {"id": "45c1f67e-53d5-420b-971c-18fa30e8d558", "name": null, "description": null, "format"
  add        part-00000-d42f10c8-2910-4079-86a7-7676e1f38ac9-c000.snappy.parquet

--- 00000000000000000001.json
  commitInfo operation=WRITE
  add        part-00000-2f7fb705-0c43-47f7-b4fa-b269168b075f-c000.snappy.parquet

--- 00000000000000000002.json
  commitInfo operation=UPDATE
  add        part-00000-1b3865cb-30ac-4474-acea-1759629b68c8-c000.snappy.parquet
  remove     part-00000-2f7fb705-0c43-47f7-b4fa-b269168b075f-c000.snappy.parquet
  remove     part-00000-d42f10c8-2910-4079-8

Versi 0 adalah keadaan tabel di akhir hari pertama. Jumlah barisnya sama dengan data tanggal 1, perjalanan terakhirnya masih di tanggal 1, dan total bayar perjalanan Gratis di versi itu belum menjadi 0. Koreksi pada hari kedua tercatat sebagai versi baru di `_delta_log/`: file lama ditandai `remove` dan file hasil koreksi ditandai `add`. File lama tetap ada di disk, sehingga keadaan akhir hari pertama tetap bisa dibaca.

In [15]:
dt = DeltaTable(PATH_DUA_HARI)
print("versi sebelum restore:", dt.version(), "| baris:", len(dt.to_pandas()))
dt.restore(1)                        # kembalikan tabel ke keadaan versi 1 (dicatat sebagai versi baru)
dt = DeltaTable(PATH_DUA_HARI)
print("versi setelah restore:", dt.version(), "| baris:", len(dt.to_pandas()))
print([h.get("operation") for h in dt.history()][:3])

versi sebelum restore: 2 | baris: 64513
versi setelah restore: 3 | baris: 64513
['RESTORE', 'UPDATE', 'WRITE']


Latihan 6: Restore
Sel ini berasal dari modul dan dijalankan sesudah K-8. Berbeda dengan time travel yang hanya membaca versi lama, restore benar-benar mengembalikan isi tabel ke keadaan versi 1. Pengembalian itu dicatat sebagai versi baru, jadi riwayat sebelumnya tetap tersimpan.